In [3]:
import re
import numpy as np
from scipy.stats import ranksums
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

class MethodComparator:
    def __init__(self):
        self.methods = {}
        self.metrics = ['auc', 'auprc']  # 支持的指标类型
    
    def parse_data_block(self, block):
        """解析数据块，支持带括号的矩阵格式和多行数字格式"""
        # 清理数据块：移除多余空格和空行
        cleaned = re.sub(r'\s+', ' ', block.strip())
        
        # 尝试解析为带括号的矩阵格式
        if cleaned.startswith('[['):
            try:
                # 创建有效的Python列表表示
                list_str = cleaned.replace(' ', ',').replace('][', '],[')
                # 使用eval安全解析（在受控环境中使用）
                array_data = np.array(eval(f'[{list_str}]'))
                return array_data.flatten()
            except Exception as e:
                # 如果解析失败，尝试其他方法
                pass
        
        # 如果带括号解析失败，尝试多行格式
        try:
            # 处理可能的多个空格分隔的数据
            numbers = [float(x) for x in cleaned.split()]
            return np.array(numbers)
        except:
            try:
                # 处理多行数据
                rows = block.strip().split('\n')
                data = []
                for row in rows:
                    # 移除行首尾空格并按空格分割
                    if row.strip():
                        # 转换为浮点数
                        row_data = [float(x) for x in re.split(r'\s+', row.strip())]
                        data.extend(row_data)
                return np.array(data)
            except Exception as e:
                raise ValueError(f"无法解析数据块: {str(e)}")
    
    def add_method(self, name, auc_data, auprc_data):
        """添加新方法的数据"""
        try:
            # 解析数据
            auc_parsed = self.parse_data_block(auc_data)
            auprc_parsed = self.parse_data_block(auprc_data)
            
            # 存储数据
            self.methods[name] = {
                'auc': auc_parsed,
                'auprc': auprc_parsed,
                'auc_mean': np.mean(auc_parsed),
                'auprc_mean': np.mean(auprc_parsed)
            }
            # print(f"成功添加方法: {name}")
            # print(f"  AUC均值: {np.mean(auc_parsed):.6f} (共 {len(auc_parsed)} 个数据点)")
            # print(f"  AUPRC均值: {np.mean(auprc_parsed):.6f} (共 {len(auprc_parsed)} 个数据点)")
            return True
        except Exception as e:
            print(f"添加方法 {name} 失败: {str(e)}")
            return False
    
    def compare_methods(self, base_method='my', alternative='greater', alpha=0.05):
        """比较基础方法与其他所有方法的性能"""
        results = {}
        
        if base_method not in self.methods:
            print(f"错误: 基础方法 '{base_method}' 不存在!")
            return None
        
        print(f"\n{'='*40}")
        print(f"将 {base_method} 与其他方法进行比较 (p < {alpha})")
        print(f"{'='*40}")
        
        # 对每个指标进行对比
        for metric in self.metrics:
            print(f"\n>>> 指标: {metric.upper()} <<<")
            base_data = self.methods[base_method][metric]
            
            pvalues = []   # ⭐ 保存所有p值，方便求平均
            
            # 与其他每个方法比较
            for method_name, method_data in self.methods.items():
                if method_name == base_method:
                    continue
                
                # 执行Wilcoxon秩和检验
                # stat, p_value = stats.ranksums(base_data, method_data[metric], alternative=alternative)
                stat, p_value = stats.wilcoxon(base_data, method_data[metric], alternative=alternative)
                pvalues.append(p_value)   # ⭐ 收集
                
                # 存储结果
                if metric not in results:
                    results[metric] = {}
                results[metric][method_name] = {
                    'statistic': stat,
                    'p_value': p_value,
                    'significant': p_value < alpha,
                    f'{base_method}_mean': np.mean(base_data),
                    f'{method_name}_mean': np.mean(method_data[metric])
                }
                
                significance = "(显著)" if p_value < alpha else "(不显著)"
                print(f"{base_method} vs {method_name:<15}: "
                      f"p值 = {p_value:.6e} {significance}")
            
            # ⭐ 在每个指标结束后输出 p 值平均值
            if pvalues:
                mean_p = np.mean(pvalues)
                results[metric]["mean_pvalue"] = mean_p
                print(f"\n>>> {metric.upper()} 平均p值 = {mean_p:.6e} <<<")
        
        return results

    def report_results(self, results, base_method='my'):
        """生成结果报告"""
        print("\n\n" + "="*60)
        print(f"{base_method.upper()} 与其他方法的比较结果总结")
        print("="*60)
        
        for metric, method_results in results.items():
            print(f"\n>>> {metric.upper()}指标 <<<")
            for method, res in method_results.items():
                # ⭐ 跳过平均值的那一项，单独处理
                if method == "mean_pvalue":
                    continue
                significance = "显著优于" if res['significant'] else "无显著差异"
                print(f" - {base_method} {significance} {method}: "
                      f"p值 = {res['p_value']:.6f}, "
                      f"{base_method}均值={res[f'{base_method}_mean']:.5f}, {method}均值={res[f'{method}_mean']:.5f}")
            
            # ⭐ 输出平均p值
            if "mean_pvalue" in method_results:
                print(f" ★ {metric.upper()} 平均p值 = {method_results['mean_pvalue']:.6e}")


    def add_new_method(self, name, auc_str, auprc_str):
        """添加新方法的便捷方法"""
        return self.add_method(name, auc_str, auprc_str)


ModuleNotFoundError: No module named 'statsmodels'

In [ ]:

# cpdb
# my_auc = """
# 0.934465 0.927817 0.928032 0.946620 0.921578
# 0.921800 0.935872 0.941748 0.918484 0.933451
# 0.931851 0.936619 0.921510 0.922024 0.943612
# 0.930042 0.952241 0.938415 0.918512 0.920642
# 0.921499 0.952241 0.939102 0.915576 0.927119
# 0.898969 0.940524 0.917978 0.955198 0.940028
# 0.914161 0.921929 0.946053 0.933379 0.930861
# 0.945765 0.909968 0.943922 0.920326 0.942734
# 0.934350 0.925332 0.938773 0.899702 0.945584
# 0.918253 0.931134 0.929276 0.951513 0.924471
# """

# my_auprc = """
# 0.859213 0.859762 0.871720 0.872537 0.833155
# 0.828864 0.881445 0.872178 0.844952 0.863403
# 0.855342 0.859985 0.837452 0.845036 0.902089
# 0.858901 0.906839 0.851249 0.845873 0.836574
# 0.861825 0.889684 0.880761 0.818253 0.852901
# 0.822867 0.867058 0.831106 0.900861 0.874264
# 0.819568 0.845461 0.893917 0.879794 0.855333
# 0.879526 0.854050 0.881020 0.852599 0.863641
# 0.825810 0.852160 0.883106 0.809234 0.891739
# 0.826929 0.871535 0.872392 0.876761 0.840643
# """

my_auc = """
0.930875 0.929095 0.928590 0.942173 0.916167
0.918742 0.939505 0.943578 0.924658 0.932602
0.930717 0.942161 0.927203 0.917217 0.939295
0.924543 0.955286 0.942620 0.917246 0.923089
0.920106 0.955616 0.944050 0.915821 0.929522
0.897145 0.934206 0.917835 0.954219 0.942662
0.913199 0.920737 0.944351 0.935308 0.928011
0.943109 0.908389 0.935870 0.924197 0.943785
0.926510 0.929497 0.938344 0.907488 0.937999
0.916774 0.929382 0.928561 0.952362 0.925464

"""

my_auprc = """
0.854515 0.862297 0.867549 0.862977 0.833417
0.823609 0.888628 0.877446 0.854587 0.862908
0.854506 0.875900 0.838501 0.835469 0.892331
0.848546 0.906891 0.869772 0.841183 0.839425
0.856996 0.897267 0.886116 0.820580 0.856040
0.814555 0.851706 0.835355 0.898456 0.878181
0.816830 0.845976 0.887433 0.873221 0.852250
0.877344 0.850972 0.869887 0.855040 0.868842
0.820411 0.863494 0.883490 0.828242 0.877284
0.828509 0.854890 0.864971 0.883431 0.843549

"""

# my_auc = '''[[0.92225956 0.93445048 0.92634439 0.93739476 0.91740426]
#  [0.92078056 0.9340197  0.93644165 0.91989407 0.92883151]
#  [0.93725051 0.93063094 0.92647311 0.91812386 0.94260467]
#  [0.91935901 0.94868039 0.93837243 0.91848366 0.91554769]
#  [0.91229431 0.94954194 0.93794336 0.90082466 0.92780968]
#  [0.89866747 0.93875822 0.91475973 0.94910985 0.93902106]
#  [0.91446254 0.91677436 0.94497998 0.93504886 0.92042658]
#  [0.94040952 0.90986761 0.93579805 0.91531742 0.93166674]
#  [0.91654461 0.9279745  0.93698513 0.90731546 0.93946721]
#  [0.91706154 0.92237443 0.92136728 0.94692227 0.92149159]]'''
# my_auprc = '''[[0.82969832 0.8620189  0.86462078 0.83673773 0.83315288]
#  [0.81837993 0.86976418 0.86411873 0.82858331 0.85431216]
#  [0.84763296 0.85796687 0.8501101  0.823092   0.89347832]
#  [0.82152208 0.88904877 0.84839531 0.85172911 0.82804811]
#  [0.84434917 0.87768454 0.87574052 0.79283924 0.83460545]
#  [0.82508383 0.85586648 0.80885169 0.88990037 0.86599608]
#  [0.8223673  0.82190522 0.88647807 0.8749761  0.80558603]
#  [0.86691776 0.83056368 0.86899248 0.84905573 0.85154994]
#  [0.78319651 0.85950249 0.87758954 0.81644966 0.86828853]
#  [0.81275027 0.84782408 0.85608705 0.86393502 0.85335758]]'''

ECD_CDGI_auc = """
[[0.90815887 0.89390023 0.88708524 0.91838291 0.87199747]
 [0.90415267 0.90178341 0.90464817 0.8748183  0.90625045]
 [0.87691336 0.90413831 0.89342105 0.87786941 0.92070003]
 [0.90587576 0.91496511 0.90676487 0.88378452 0.87631507]
 [0.87531949 0.92411189 0.9090389  0.86423298 0.88522372]
 [0.87183022 0.89667155 0.88116419 0.92170747 0.90763208]
 [0.88775452 0.88987967 0.90956808 0.90817898 0.89079343]
 [0.90380805 0.89543666 0.9076659  0.88749766 0.88564109]
 [0.89527871 0.88201086 0.90829519 0.88067585 0.91248219]
 [0.89051147 0.90706757 0.88495423 0.91409409 0.88412993]]
"""

ECD_CDGI_auprc = """
[[0.82237284 0.8153618  0.80137353 0.82283677 0.78989792]
 [0.82926049 0.82204888 0.80716318 0.76804461 0.83277482]
 [0.75933082 0.8216658  0.79298996 0.78823325 0.84826952]
 [0.82085703 0.84280922 0.80469405 0.80347888 0.7817941 ]
 [0.79853734 0.85542321 0.82120369 0.75185585 0.76009824]
 [0.77458019 0.80755138 0.78765343 0.84119816 0.82594477]
 [0.78290548 0.79247823 0.84460316 0.8377688  0.79474446]
 [0.80902245 0.81547531 0.83527141 0.79863394 0.78465161]
 [0.77883604 0.78399173 0.84066164 0.77160458 0.83747113]
 [0.7774311  0.83562438 0.80218094 0.82150933 0.79012029]]
"""

dishyper_auc = """
0.889939 0.938110 0.870020 0.869004 0.915346
0.908333 0.872459 0.910874 0.875711 0.891565
0.977947 0.870427 0.836992 0.852947 0.917378
0.915346 0.899695 0.865549 0.895325 0.855894
0.894715 0.884248 0.932012 0.901524 0.783841
0.859146 0.927744 0.893699 0.914634 0.845224
0.897358 0.940142 0.861992 0.871443 0.905081
0.907419 0.917988 0.895020 0.920833 0.859654
0.874289 0.947764 0.877439 0.902236 0.847053
0.899187 0.866362 0.939939 0.819512 0.864126
"""

dishyper_auprc = """
0.583120 0.739021 0.602123 0.651525 0.687891
0.642072 0.567824 0.608731 0.692058 0.634089
0.848423 0.612691 0.457410 0.522460 0.663062
0.752783 0.550915 0.541286 0.680000 0.572206
0.626262 0.554535 0.792559 0.627898 0.435688
0.606642 0.763636 0.571991 0.642631 0.619308
0.645200 0.710110 0.566075 0.687758 0.762122
0.609563 0.757472 0.702980 0.615460 0.531874
0.603039 0.754188 0.643845 0.672967 0.627290
0.551683 0.520233 0.700800 0.597828 0.557613
"""

DISFusion_auc = """
0.928592 0.928664 0.925072 0.931235 0.907243
0.918311 0.929037 0.934039 0.918052 0.916411
0.924255 0.927702 0.916991 0.922240 0.937380
0.919890 0.946785 0.936885 0.918815 0.901515
0.914089 0.949642 0.927646 0.896737 0.920110
0.886591 0.935154 0.922411 0.949038 0.935006
0.905574 0.924399 0.931894 0.929422 0.925277
0.928850 0.905904 0.937228 0.923032 0.937222
0.916329 0.907585 0.944351 0.906682 0.934588
0.912639 0.929051 0.916347 0.939669 0.910453
"""

DISFusion_auprc = """
0.845590 0.854397 0.851754 0.843431 0.818816
0.811549 0.864555 0.854627 0.837381 0.841629
0.843386 0.869397 0.824175 0.826804 0.885200
0.841961 0.886561 0.851219 0.828152 0.814797
0.831762 0.872183 0.853374 0.802326 0.846439
0.795617 0.848709 0.832553 0.886959 0.865262
0.808896 0.844638 0.861581 0.852587 0.849845
0.848850 0.840051 0.856907 0.854808 0.846855
0.804869 0.829399 0.885059 0.807518 0.871639
0.820426 0.856796 0.827002 0.851622 0.831335
"""
MNGCL_auc = """[[0.88686484 0.93960874 0.93507891 0.91056057 0.93121029]
 [0.88584858 0.93577236 0.93431031 0.91335315 0.92362677]
 [0.8769563  0.93856707 0.93730785 0.91109859 0.92083419]
 [0.88579776 0.93541667 0.93548883 0.91153413 0.92967309]
 [0.88249492 0.93216463 0.93564255 0.90748617 0.92393421]
 [0.88376524 0.9351372  0.93341361 0.91009941 0.92757225]
 [0.88485772 0.93246951 0.93602685 0.91509531 0.92823837]
 [0.87898882 0.93836382 0.93618057 0.90789609 0.92347305]
 [0.88041159 0.93818598 0.93413097 0.90799857 0.93026235]
 [0.88127541 0.94288618 0.93395163 0.91209777 0.93003177]]
"""
MNGCL_auprc = """[[0.81463485 0.86920769 0.84275703 0.82929953 0.86415289]
 [0.81146955 0.86690593 0.84233846 0.82729229 0.85039867]
 [0.8019149  0.86506983 0.84568499 0.82465973 0.84900021]
 [0.81285456 0.8624229  0.83974432 0.8270131  0.86161068]
 [0.80721677 0.86264949 0.84189057 0.81857494 0.85099538]
 [0.80650452 0.86191705 0.84326771 0.82481034 0.85340206]
 [0.80547093 0.85799116 0.84313832 0.83806408 0.85519645]
 [0.803481   0.87534672 0.84086712 0.8234699  0.85453589]
 [0.80514657 0.86696241 0.84160703 0.82517066 0.85637534]
 [0.80420265 0.88036771 0.84648552 0.82834985 0.85646726]]
"""
MTGCN_auc = """0.9200 0.9114 0.9058 0.9296 0.8935
0.9131 0.9129 0.9176 0.9013 0.8962
0.9114 0.9191 0.9062 0.8983 0.9262
0.8961 0.9479 0.9107 0.9027 0.8910
0.8990 0.9334 0.9178 0.8857 0.8986
0.8822 0.9151 0.8928 0.9331 0.9222
0.8933 0.8950 0.9281 0.9190 0.9051
0.9251 0.8966 0.9233 0.9002 0.9261
0.9016 0.9006 0.9116 0.8904 0.9251
0.8951 0.9226 0.8963 0.9423 0.8917
"""
MTGCN_auprc = """0.8320 0.8507 0.8418 0.8583 0.7983
0.8234 0.8524 0.8383 0.8124 0.8282
0.8301 0.8571 0.8104 0.8129 0.8708
0.8246 0.8984 0.8154 0.8184 0.7984
0.8338 0.8791 0.8433 0.7713 0.8124
0.8174 0.8324 0.7943 0.8624 0.8481
0.8103 0.8134 0.8636 0.8347 0.8244
0.8478 0.8270 0.8566 0.8256 0.8476
0.8043 0.8135 0.8487 0.8047 0.8726
0.8145 0.8605 0.8103 0.8688 0.8108
"""
EMOGI_auc = """
[[0.9082881  0.89349818 0.89294908 0.92159233 0.8868788 ]
 [0.90896298 0.90179777 0.91118421 0.88640387 0.87709943]
 [0.90102237 0.90533012 0.88536899 0.89044802 0.91809507]
 [0.89122943 0.92932426 0.90061499 0.89918397 0.88767037]
 [0.8805893  0.92721346 0.8951087  0.8708605  0.89620483]
 [0.86653169 0.90811579 0.89041762 0.91920326 0.90816459]
 [0.89647052 0.87564975 0.91880721 0.91351841 0.89809018]
 [0.90535884 0.87319434 0.91029748 0.89669416 0.90619288]
 [0.88778323 0.88597398 0.91812071 0.88510859 0.90757451]
 [0.89311048 0.91309842 0.88913043 0.92320424 0.87582574]]
"""
EMOGI_auprc = """[[0.82455664 0.82515941 0.81143053 0.84570757 0.78353942]
 [0.81911463 0.83507479 0.82530041 0.78496981 0.79072934]
 [0.81595928 0.82723738 0.79677961 0.80291758 0.85589096]
 [0.81541817 0.86732127 0.78731786 0.81403892 0.80204327]
 [0.81238975 0.85105359 0.81165827 0.7552621  0.80256303]
 [0.79072179 0.81812382 0.79224141 0.8454298  0.80865332]
 [0.80780765 0.78734652 0.85881486 0.82829197 0.80631242]
 [0.82224823 0.80043445 0.84394804 0.81823899 0.81637219]
 [0.7902541  0.78594499 0.85535691 0.79094149 0.83082377]
 [0.80472951 0.84791518 0.80475559 0.83721107 0.76752404]]
"""
GCN_auc = """[[0.91170558 0.89839465 0.90170195 0.91937596 0.87722177]
 [0.90563166 0.91541024 0.91394451 0.88568427 0.89198797]
 [0.89704489 0.90485626 0.891004   0.88541082 0.92044097]
 [0.89378536 0.93587203 0.9007008  0.88430264 0.89688125]
 [0.87938313 0.93934695 0.91049771 0.87520689 0.89857951]
 [0.86851325 0.90995376 0.88468249 0.92942159 0.91682858]
 [0.88983659 0.88979352 0.91812071 0.90555963 0.90085345]
 [0.91518049 0.87576462 0.9159468  0.89948621 0.89259243]
 [0.89876798 0.88640476 0.91651888 0.88243167 0.91707324]
 [0.90023262 0.91450561 0.89562357 0.92120375 0.87863218]]
"""

GCN_auprc = """[[0.8063275  0.80920504 0.83337165 0.83090988 0.78656972]
 [0.80885014 0.84070705 0.83351776 0.77775157 0.81267421]
 [0.80937057 0.83378759 0.77621606 0.7957097  0.85528141]
 [0.80876723 0.87959626 0.79437608 0.78798432 0.79264129]
 [0.80716571 0.8763009  0.83073968 0.76167223 0.80069932]
 [0.79043977 0.82769211 0.78889572 0.84736591 0.82606097]
 [0.77743507 0.8013346  0.85175804 0.83089457 0.81187919]
 [0.82132415 0.80381993 0.83324138 0.81150627 0.80385297]
 [0.77496287 0.79498822 0.84859233 0.78119553 0.84764909]
 [0.8099716  0.8320813  0.81374254 0.84245598 0.78128406]]
"""

Cheb_auc = """[[0.9144769  0.90089314 0.90308924 0.92259977 0.88064706]
 [0.90283162 0.90788605 0.91623284 0.8903041  0.89285149]
 [0.89405818 0.89945722 0.90673627 0.88885051 0.92342012]
 [0.89639872 0.9424198  0.90311785 0.88365499 0.88883612]
 [0.8870222  0.94289366 0.90799485 0.87707785 0.88899443]
 [0.87712874 0.90955171 0.88735698 0.93306276 0.90945987]
 [0.89239252 0.88999454 0.91784897 0.90725789 0.89747132]
 [0.91395997 0.87857902 0.91792048 0.90122764 0.89217506]
 [0.89427357 0.89681514 0.91368707 0.8805895  0.91831095]
 [0.89825106 0.91568306 0.89043192 0.92377992 0.88217262]]
"""
Cheb_auprc = """[[0.82064163 0.82428834 0.83030134 0.8362699  0.77737436]
 [0.79899852 0.84177632 0.82949555 0.77812934 0.82197329]
 [0.79553466 0.82392843 0.79850235 0.80758621 0.85981662]
 [0.81277372 0.88345712 0.79367242 0.78691337 0.78735599]
 [0.80963331 0.87677303 0.83170113 0.76108116 0.79707984]
 [0.80409972 0.81425681 0.79319137 0.85585175 0.83046795]
 [0.78192137 0.79561317 0.85029353 0.82191211 0.81475644]
 [0.81798088 0.8037637  0.84147641 0.81086011 0.80682255]
 [0.76566652 0.80579753 0.84798188 0.78535385 0.84933169]
 [0.79879291 0.8466793  0.80403991 0.84077089 0.78334198]]
"""

GAT_auc = """[[0.82038138 0.80929611 0.77325515 0.80665774 0.81370983]
 [0.78871945 0.81488182 0.82261156 0.78085287 0.83522588]
 [0.79620057 0.82891071 0.80267449 0.79386325 0.84878316]
 [0.79599954 0.86047213 0.81846396 0.79816646 0.80618281]
 [0.80056575 0.85192843 0.79453661 0.79636746 0.81395449]
 [0.80167141 0.80320783 0.79656751 0.8478045  0.82362592]
 [0.79920163 0.8069125  0.78481121 0.83388743 0.83816185]
 [0.85222998 0.78660866 0.81161327 0.81737979 0.82174057]
 [0.83283076 0.80580684 0.83302346 0.81077386 0.83603184]
 [0.81733724 0.82555067 0.80015732 0.84065887 0.81627161]]
"""
GAT_auprc = """[[0.61432786 0.57383708 0.51140274 0.58317297 0.55059964]
 [0.54190839 0.62071861 0.60847821 0.49487474 0.63683661]
 [0.50094427 0.63525247 0.59761897 0.50473188 0.66339336]
 [0.56507244 0.6612198  0.55180366 0.55459829 0.5753511 ]
 [0.56338371 0.66373491 0.53135221 0.52795286 0.589724  ]
 [0.57793828 0.55412917 0.5697428  0.63354895 0.5624003 ]
 [0.58815425 0.5725286  0.50309343 0.61207066 0.63170266]
 [0.64472187 0.53278857 0.55298839 0.59953201 0.55450752]
 [0.56082553 0.61194922 0.63948537 0.55935936 0.62090876]
 [0.55303743 0.6006368  0.54670427 0.57404289 0.63619995]]
"""
# ===== 使用示例 =====
if __name__ == "__main__":
    # 创建比较器
    comparator = MethodComparator()
    
    # 添加方法数据
    comparator.add_method('my', my_auc, my_auprc)

    comparator.add_method('ECD_CDGI', ECD_CDGI_auc, ECD_CDGI_auprc)
    comparator.add_method('Dishyper', dishyper_auc, dishyper_auprc)
    comparator.add_method('DISFusion', DISFusion_auc, DISFusion_auprc)
    comparator.add_method('MNGCL', MNGCL_auc, MNGCL_auprc)
    comparator.add_method('MTGCN', MTGCN_auc, MTGCN_auprc)
    comparator.add_method('EMOGI', EMOGI_auc, EMOGI_auprc)

    comparator.add_method('GCN', GCN_auc, GCN_auprc)
    comparator.add_method('Cheb', Cheb_auc, Cheb_auprc)
    comparator.add_method('GAT', GAT_auc, GAT_auprc)
    
    # 比较方法（默认比较my方法与其他方法）
    results = comparator.compare_methods(alternative='greater', alpha=0.05)
    
    # 生成报告
    comparator.report_results(results)
    results_cpdb = results

    


In [ ]:

# string
# my_auc = """
# 0.936820 0.930545 0.913687 0.939208 0.917519
# 0.934379 0.931062 0.936957 0.925449 0.919304
# 0.927917 0.932842 0.926416 0.923780 0.941914
# 0.920379 0.958129 0.939231 0.926414 0.916829
# 0.918756 0.953304 0.935884 0.904869 0.922945
# 0.898667 0.943382 0.908753 0.945742 0.938086
# 0.912050 0.912596 0.940275 0.939482 0.929220
# 0.941099 0.901238 0.935955 0.920196 0.935768
# 0.918009 0.923882 0.934110 0.914814 0.938316
# 0.919546 0.928233 0.926630 0.944720 0.924428
# """

# my_auprc = """
# 0.844804 0.860210 0.851963 0.840068 0.831133
# 0.838773 0.873166 0.847820 0.844351 0.843606
# 0.849770 0.866395 0.846214 0.822312 0.885173
# 0.821214 0.902823 0.852345 0.848871 0.841517
# 0.857184 0.885990 0.871910 0.796042 0.834543
# 0.810505 0.877395 0.808855 0.886323 0.860581
# 0.816121 0.804948 0.882203 0.875449 0.838845
# 0.858770 0.809200 0.870844 0.841090 0.852504
# 0.812342 0.853273 0.880602 0.827878 0.872644
# 0.807144 0.875804 0.846418 0.861213 0.853950
# """
# my_auc = """
# [[0.92079756 0.92340863 0.93554087 0.92660721 0.91616234]
#  [0.91917554 0.92528122 0.9343804  0.9313585  0.92194882]
#  [0.91751395 0.92310533 0.93444633 0.92813346 0.91779477]
#  [0.92350094 0.92720655 0.93563318 0.92561183 0.92289112]
#  [0.91893817 0.92388337 0.93037148 0.92801401 0.91535276]
#  [0.91862167 0.92574277 0.93030555 0.92538621 0.91742316]
#  [0.91565455 0.92238003 0.93333861 0.93036312 0.91794075]
#  [0.91039285 0.9206525  0.93198032 0.92533312 0.91705155]
#  [0.92247234 0.92627026 0.93378697 0.92773531 0.91893614]
#  [0.91550949 0.9237515  0.92741755 0.92774858 0.91924139]]
# """

# my_auprc = """
# [[0.8342188  0.82700367 0.84427545 0.83879462 0.794123  ]
#  [0.82905702 0.82546322 0.83826925 0.84711287 0.80303074]
#  [0.8271907  0.81991264 0.83854856 0.82656543 0.80068012]
#  [0.81510998 0.82751142 0.84477215 0.81611358 0.82428673]
#  [0.83506651 0.8253234  0.84111714 0.8367867  0.78629465]
#  [0.83045839 0.8247528  0.83857889 0.83232829 0.79100537]
#  [0.82589099 0.81912867 0.84374997 0.83948731 0.79714966]
#  [0.81129264 0.8140708  0.82717354 0.83194359 0.79849702]
#  [0.83352383 0.82517313 0.84892627 0.82732134 0.80269248]
#  [0.82444379 0.82343009 0.82865713 0.83823748 0.80161444]]
# """
my_auc = """
0.926850 0.927932 0.940341 0.928930 0.923011
0.923184 0.921998 0.946473 0.923820 0.923714
0.920995 0.921866 0.943598 0.927603 0.923621
0.925571 0.926125 0.945735 0.926196 0.922055
0.927048 0.929620 0.941963 0.927921 0.917742
0.918674 0.929897 0.940512 0.926899 0.925798
0.922222 0.928116 0.943005 0.928160 0.923940
0.925505 0.927167 0.940644 0.927404 0.926329
0.926244 0.929791 0.943229 0.927271 0.929195
0.926099 0.927048 0.946236 0.928372 0.923037
"""

my_auprc = """
0.837263 0.843070 0.868576 0.853001 0.815855
0.835163 0.832547 0.877656 0.844799 0.825711
0.829750 0.826577 0.872899 0.841344 0.820794
0.840209 0.838871 0.877213 0.846903 0.821167
0.838712 0.843895 0.867904 0.857366 0.817079
0.826958 0.847139 0.871516 0.850270 0.822796
0.837267 0.841566 0.871689 0.849761 0.818099
0.836109 0.839406 0.870294 0.849378 0.821353
0.843582 0.842520 0.869686 0.851892 0.831867
0.837125 0.832199 0.875197 0.851124 0.823483
"""

ECD_CDGI_auc = """
[[0.89158787 0.89931558 0.89472643 0.91544567 0.87098529]
 [0.88947792 0.90510477 0.89975076 0.90724372 0.86964485]
 [0.88335905 0.90012    0.90182115 0.90624834 0.86636673]
 [0.89265604 0.89959251 0.89807598 0.91507406 0.868822  ]
 [0.88404478 0.9042476  0.90617294 0.90285077 0.86432288]
 [0.89154831 0.89973757 0.90099036 0.90618198 0.86673833]
 [0.8886735  0.90331131 0.90062112 0.91482189 0.86712321]
 [0.8860888  0.90521027 0.90430035 0.90861071 0.87468811]
 [0.89078345 0.90319263 0.9079664  0.90497425 0.86827786]
 [0.88900318 0.90017275 0.89993538 0.9079604  0.86709667]]
"""

ECD_CDGI_auprc = """
[[0.76029614 0.74457696 0.74416887 0.81692398 0.70453444]
 [0.75761678 0.76282806 0.7704174  0.79938258 0.7105372 ]
 [0.75760313 0.73564764 0.77629384 0.78296617 0.70804169]
 [0.75987127 0.73720541 0.7677653  0.80715499 0.71110197]
 [0.74798142 0.7590439  0.78347975 0.78549603 0.70093629]
 [0.76319297 0.75160366 0.76001649 0.78888787 0.70269432]
 [0.75342343 0.74237831 0.77862225 0.80730674 0.71085086]
 [0.7512087  0.7544904  0.76258612 0.78019729 0.72469316]
 [0.7563112  0.76106424 0.78304616 0.7878909  0.70072683]
 [0.75805732 0.75477802 0.76723166 0.80536726 0.70059727]]
"""

dishyper_auc = """
0.861099 0.876423 0.883254 0.892340 0.866606
0.864673 0.880761 0.878651 0.893653 0.865610
0.861376 0.884942 0.876871 0.894981 0.865677
0.863684 0.885588 0.882014 0.894450 0.865982
0.862945 0.884177 0.880946 0.898113 0.866261
0.863301 0.880735 0.882370 0.891225 0.866446
0.863499 0.887553 0.884335 0.888823 0.866964
0.866440 0.876607 0.882832 0.892711 0.867216
0.862550 0.883214 0.878651 0.892924 0.866393
0.863341 0.885944 0.883161 0.889884 0.866247
"""

dishyper_auprc = """
0.751763 0.724373 0.762033 0.777284 0.728459
0.763390 0.731180 0.744492 0.779115 0.724890
0.754597 0.745736 0.744112 0.783877 0.724847
0.760335 0.748106 0.758435 0.782760 0.721712
0.758879 0.741999 0.757763 0.795775 0.723778
0.758036 0.731803 0.761580 0.775644 0.735384
0.757674 0.753874 0.767414 0.767684 0.741054
0.766458 0.721360 0.759656 0.778240 0.735453
0.756908 0.743176 0.746147 0.780292 0.724938
0.758739 0.749929 0.763843 0.770000 0.724416
"""

DISFusion_auc = """
0.917830 0.927787 0.923791 0.941697 0.908478
0.943163 0.899276 0.919795 0.920197 0.947510
0.922143 0.936504 0.924450 0.917967 0.919494
0.936504 0.937835 0.920428 0.917091 0.917901
0.919795 0.934394 0.924121 0.919746 0.934371
0.920758 0.925083 0.926046 0.938990 0.910933
0.919848 0.905065 0.925875 0.931690 0.947205
0.930292 0.903826 0.939339 0.929912 0.925731
0.923514 0.939550 0.904696 0.942560 0.917954
0.923593 0.927615 0.934354 0.918113 0.918060
"""

DISFusion_auprc = """
0.812532 0.821237 0.844231 0.856270 0.799276
0.861749 0.794392 0.825837 0.818084 0.870755
0.831525 0.829849 0.835080 0.806674 0.815372
0.850873 0.844970 0.810728 0.825225 0.827401
0.821650 0.849317 0.820772 0.811598 0.841893
0.807026 0.835011 0.833040 0.838668 0.824004
0.826198 0.800092 0.817203 0.838821 0.866501
0.821916 0.801878 0.863688 0.835631 0.823854
0.817249 0.829365 0.792719 0.862634 0.835423
0.808579 0.844128 0.857633 0.802224 0.816768
"""
MNGCL_auc = """[[0.9223941  0.92206667 0.9151606  0.91571044 0.9225811 ]
 [0.92338301 0.92564847 0.91399007 0.91693819 0.91814232]
 [0.92368911 0.92445454 0.91052533 0.91852009 0.92059782]
 [0.9195451  0.92185598 0.91132128 0.91124805 0.91887425]
 [0.93190648 0.92117708 0.91466898 0.91379799 0.92140058]
 [0.92456029 0.92232419 0.9084418  0.91814232 0.92125891]
 [0.9260672  0.92623373 0.91190655 0.91762289 0.92310053]
 [0.92750347 0.92361176 0.9088866  0.91700902 0.9196534 ]
 [0.92423065 0.92354153 0.91092331 0.91280635 0.91679652]
 [0.92905747 0.92450136 0.91782938 0.91334939 0.92336025]]
"""
MNGCL_auprc = """[[0.82155472 0.81698021 0.81281728 0.80460322 0.82593252]
 [0.83205962 0.82683861 0.80772965 0.8034747  0.81980556]
 [0.82915609 0.82471864 0.80192102 0.81229376 0.82404222]
 [0.82643151 0.81816865 0.80355006 0.78730467 0.81699728]
 [0.84719852 0.81039709 0.81457848 0.79344383 0.82138741]
 [0.83411882 0.81282232 0.79642497 0.79977525 0.8206868 ]
 [0.83163658 0.82315286 0.80563492 0.80254676 0.82582951]
 [0.83703602 0.82291167 0.80695401 0.79714068 0.82073013]
 [0.83898976 0.81417348 0.81155413 0.80203    0.81568646]
 [0.84129473 0.8143374  0.8039884  0.80240573 0.83074508]]
"""
MTGCN_auc = """9.068850470124355523e-01	9.190172884440401813e-01	9.360419881051286195e-01	9.184052662313532123e-01	9.194537346711260106e-01
9.115269480819190306e-01	9.193733433556197099e-01	9.309385343724861928e-01	9.174762435632000113e-01	9.176222328396240746e-01
9.091400614524403068e-01	9.212854901029922106e-01	9.337342247893342106e-01	9.197855284811806698e-01	9.159234485321442465e-01
9.106434044124434024e-01	9.198217087998311614e-01	9.333649826588071852e-01	9.160827095609704873e-01	9.154191219408610580e-01
9.087840065408606671e-01	9.177249409871952590e-01	9.353826271577586615e-01	9.202898550724638582e-01	9.189494080798428222e-01
9.123841173134996652e-01	9.219184766124671748e-01	9.347364534293363114e-01	9.185777990125816306e-01	9.159367202845464240e-01
9.093906186124408597e-01	9.176326304545634471e-01	9.337474120082815965e-01	9.183123639645379699e-01	9.162021553325900847e-01
9.071487913913836021e-01	9.211404306945707443e-01	9.342221518903879307e-01	9.193342888995063023e-01	9.147555343207516287e-01
9.085334493808601142e-01	9.200722659598318254e-01	9.348287639619680123e-01	9.178345808780591364e-01	9.175558740776132982e-01
9.117775052419194726e-01	9.178568131766692284e-01	9.350265722461790219e-01	9.179274831448743788e-01	9.167197536762753396e-01
"""
MTGCN_auprc = """8.065628751230571325e-01	8.097541815467227400e-01	8.606168491942842724e-01	8.336626413652389322e-01	8.224469585283959994e-01
8.141682720341660850e-01	8.141256691577967164e-01	8.533833778674260184e-01	8.338768734298076524e-01	8.120506647874136119e-01
8.144362541349947637e-01	8.183323598980094538e-01	8.543609854752778343e-01	8.350442981545430632e-01	8.078702198687095493e-01
8.166959056024669650e-01	8.143369108855905703e-01	8.498884671157826975e-01	8.288711603728514676e-01	8.155253144534964083e-01
8.119570409741191375e-01	8.090516774701981273e-01	8.604787286945571045e-01	8.345451793103857430e-01	8.151403365220875052e-01
8.163028167024326542e-01	8.176259508010477184e-01	8.431305276990551656e-01	8.306072696679018819e-01	8.155967913672712566e-01
8.159138268610279798e-01	8.148640360086506496e-01	8.469829586857422488e-01	8.336196733962140115e-01	8.078597535767736426e-01
8.137096115425591636e-01	8.179994420079207407e-01	8.547231396628875988e-01	8.328470963818928041e-01	8.152616278804867278e-01
8.112382824165612627e-01	8.126720812210316280e-01	8.437344687584624747e-01	8.265493407736124176e-01	8.179740845100474944e-01
8.128977512643433823e-01	8.091638181206525227e-01	8.496864891873477932e-01	8.283613268858146528e-01	8.119420309971184802e-01
"""
EMOGI_auc = """
[[0.9082881  0.89349818 0.89294908 0.92159233 0.8868788 ]
 [0.90896298 0.90179777 0.91118421 0.88640387 0.87709943]
 [0.90102237 0.90533012 0.88536899 0.89044802 0.91809507]
 [0.89122943 0.92932426 0.90061499 0.89918397 0.88767037]
 [0.8805893  0.92721346 0.8951087  0.8708605  0.89620483]
 [0.86653169 0.90811579 0.89041762 0.91920326 0.90816459]
 [0.89647052 0.87564975 0.91880721 0.91351841 0.89809018]
 [0.90535884 0.87319434 0.91029748 0.89669416 0.90619288]
 [0.88778323 0.88597398 0.91812071 0.88510859 0.90757451]
 [0.89311048 0.91309842 0.88913043 0.92320424 0.87582574]]
"""
EMOGI_auprc = """[[0.82455664 0.82515941 0.81143053 0.84570757 0.78353942]
 [0.81911463 0.83507479 0.82530041 0.78496981 0.79072934]
 [0.81595928 0.82723738 0.79677961 0.80291758 0.85589096]
 [0.81541817 0.86732127 0.78731786 0.81403892 0.80204327]
 [0.81238975 0.85105359 0.81165827 0.7552621  0.80256303]
 [0.79072179 0.81812382 0.79224141 0.8454298  0.80865332]
 [0.80780765 0.78734652 0.85881486 0.82829197 0.80631242]
 [0.82224823 0.80043445 0.84394804 0.81823899 0.81637219]
 [0.7902541  0.78594499 0.85535691 0.79094149 0.83082377]
 [0.80472951 0.84791518 0.80475559 0.83721107 0.76752404]]
"""
GCN_auc = """[[0.81983621 0.84296    0.86482441 0.85912698 0.81821681]
 [0.83335971 0.84690298 0.86945313 0.85444206 0.82750703]
 [0.83117063 0.84942174 0.85488784 0.86334077 0.82055264]
 [0.83241023 0.8524548  0.87553903 0.85336704 0.82896693]
 [0.82512429 0.84187865 0.86242434 0.8432606  0.83156819]
 [0.83268057 0.84762828 0.86695415 0.84369194 0.81899984]
 [0.8352257  0.83830491 0.87491263 0.85895445 0.82335298]
 [0.83011565 0.82696391 0.87375875 0.85075914 0.82537028]
 [0.83411138 0.84495127 0.8843415  0.85414344 0.83009503]
 [0.82898814 0.84730519 0.8673168  0.86206004 0.81335935]]
"""

GCN_auprc = """[[0.56443823 0.58743987 0.64221389 0.59775235 0.51453285]
 [0.57163883 0.58167549 0.64415955 0.58086328 0.54879213]
 [0.59496397 0.57757842 0.63061554 0.59705612 0.53048663]
 [0.58135932 0.60468769 0.66672334 0.56939242 0.58032252]
 [0.54465896 0.57744447 0.6409871  0.57814583 0.54912294]
 [0.58945842 0.58056172 0.66041311 0.56368179 0.52023213]
 [0.57579334 0.55612185 0.66948251 0.60754846 0.56344855]
 [0.55468268 0.53412999 0.66625315 0.59432677 0.52060665]
 [0.57302606 0.59477174 0.71892535 0.61432544 0.55278257]
 [0.58483013 0.59717225 0.64590191 0.61017698 0.51406136]]
"""

Cheb_auc = """[[0.89472643 0.9039443  0.92191848 0.91884323 0.89912141]
 [0.89124501 0.90521027 0.9255054  0.91650741 0.90143069]
 [0.89505611 0.90705648 0.92409437 0.91742316 0.90104581]
 [0.89679682 0.90406298 0.92206354 0.91849817 0.9016165 ]
 [0.89033509 0.90585644 0.92816922 0.91967935 0.8976217 ]
 [0.89102082 0.90511796 0.92615157 0.91795403 0.89263152]
 [0.89241867 0.90515752 0.92948794 0.92015714 0.90311621]
 [0.89183843 0.90944337 0.92050744 0.91759569 0.8988427 ]
 [0.89746937 0.90836202 0.92959344 0.91725062 0.89467537]
 [0.89491105 0.90366737 0.9235405  0.91747624 0.90153687]]
"""
Cheb_auprc = """[[0.79662332 0.78996658 0.83153267 0.83376129 0.77611994]
 [0.79599657 0.79483849 0.83192309 0.82998568 0.77759737]
 [0.79616425 0.80024771 0.83089947 0.82950815 0.77863038]
 [0.80165273 0.79436118 0.82784204 0.83603204 0.77678327]
 [0.79109808 0.8002981  0.83122676 0.83647686 0.76989367]
 [0.79402948 0.79434017 0.83191041 0.83180272 0.76647945]
 [0.79333652 0.79348081 0.84006984 0.83372893 0.78176533]
 [0.79513823 0.80108261 0.82352269 0.8356133  0.77792491]
 [0.8019772  0.79955654 0.83611097 0.83138959 0.76746541]
 [0.79593882 0.79485633 0.82975196 0.83042711 0.78165496]]
"""

GAT_auc = """[[0.82038138 0.80929611 0.77325515 0.80665774 0.81370983]
 [0.78871945 0.81488182 0.82261156 0.78085287 0.83522588]
 [0.79620057 0.82891071 0.80267449 0.79386325 0.84878316]
 [0.79599954 0.86047213 0.81846396 0.79816646 0.80618281]
 [0.80056575 0.85192843 0.79453661 0.79636746 0.81395449]
 [0.80167141 0.80320783 0.79656751 0.8478045  0.82362592]
 [0.79920163 0.8069125  0.78481121 0.83388743 0.83816185]
 [0.85222998 0.78660866 0.81161327 0.81737979 0.82174057]
 [0.83283076 0.80580684 0.83302346 0.81077386 0.83603184]
 [0.81733724 0.82555067 0.80015732 0.84065887 0.81627161]]
"""
GAT_auprc = """[[0.61432786 0.57383708 0.51140274 0.58317297 0.55059964]
 [0.54190839 0.62071861 0.60847821 0.49487474 0.63683661]
 [0.50094427 0.63525247 0.59761897 0.50473188 0.66339336]
 [0.56507244 0.6612198  0.55180366 0.55459829 0.5753511 ]
 [0.56338371 0.66373491 0.53135221 0.52795286 0.589724  ]
 [0.57793828 0.55412917 0.5697428  0.63354895 0.5624003 ]
 [0.58815425 0.5725286  0.50309343 0.61207066 0.63170266]
 [0.64472187 0.53278857 0.55298839 0.59953201 0.55450752]
 [0.56082553 0.61194922 0.63948537 0.55935936 0.62090876]
 [0.55303743 0.6006368  0.54670427 0.57404289 0.63619995]]
"""
# ===== 使用示例 =====
if __name__ == "__main__":
    # 创建比较器
    comparator = MethodComparator()
    
    # 添加方法数据
    comparator.add_method('my', my_auc, my_auprc)

    comparator.add_method('ECD_CDGI', ECD_CDGI_auc, ECD_CDGI_auprc)
    comparator.add_method('Dishyper', dishyper_auc, dishyper_auprc)
    comparator.add_method('DISFusion', DISFusion_auc, DISFusion_auprc)
    comparator.add_method('MNGCL', MNGCL_auc, MNGCL_auprc)
    comparator.add_method('MTGCN', MTGCN_auc, MTGCN_auprc)
    comparator.add_method('EMOGI', EMOGI_auc, EMOGI_auprc)

    comparator.add_method('GCN', GCN_auc, GCN_auprc)
    comparator.add_method('Cheb', Cheb_auc, Cheb_auprc)
    comparator.add_method('GAT', GAT_auc, GAT_auprc)
    
    # 比较方法（默认比较my方法与其他方法）
    results = comparator.compare_methods(alternative='greater', alpha=0.05)
    
    # 生成报告
    comparator.report_results(results)
    results_string = results

    


In [ ]:
import numpy as np
from scipy import stats
import re

class MethodComparator:
    def __init__(self):
        self.methods = {}
        self.metrics = ['auc', 'auprc']
    
    def add_method(self, name, auc_data, auprc_data):
        """添加方法数据"""
        self.methods[name] = {
            'auc': np.array(auc_data),
            'auprc': np.array(auprc_data),
            'auc_mean': np.mean(auc_data),
            'auprc_mean': np.mean(auprc_data)
        }

    def compare_against(self, target_method='BIMODriver', alpha=0.05):
        """执行 Wilcoxon 符号秩检验 (配对样本)"""
        if target_method not in self.methods:
            print(f"错误: 找不到方法 {target_method}")
            return

        print(f"{'='*80}")
        print(f"显著性分析报告: {target_method} vs 其他方法 (Alpha={alpha})")
        print(f"{'='*80}")

        for metric in self.metrics:
            print(f"\n[ 指标: {metric.upper()} ]")
            print(f"{'对比方法':<15} | {'p-value':<12} | {'显著性':<10} | {target_method}均值 | 对比方法均值")
            print("-" * 80)
            
            target_data = self.methods[target_method][metric]
            p_values = []

            for name, data in self.methods.items():
                if name == target_method:
                    continue
                
                # 使用 wilcoxon 符号秩检验（针对同一组癌症的不同模型表现）
                # alternative='greater' 表示检验 target 是否显著大于对方
                stat, p_val = stats.wilcoxon(target_data, data[metric], alternative='greater')
                p_values.append(p_val)
                
                is_sig = "YES" if p_val < alpha else "NO"
                target_mean = self.methods[target_method][f'{metric}_mean']
                other_mean = data[f'{metric}_mean']
                
                print(f"{name:<15} | {p_val:.6e} | {is_sig:<10} | {target_mean:.4f}      | {other_mean:.4f}")
            
            print(f"\n>>> {metric.upper()} 平均 p-value: {np.mean(p_values):.6e}")

# ===== 数据解析与执行 =====

# 从表格中提取的数据 (15个癌症类型对应的值)
data = {
    "GAT":      [[0.8380, 0.8399, 0.8445, 0.7309, 0.8043, 0.7603, 0.7702, 0.7461, 0.8096, 0.7667, 0.8749, 0.8270, 0.7652, 0.7709, 0.8124],
                 [0.1626, 0.3146, 0.0346, 0.1290, 0.1706, 0.1603, 0.0547, 0.0293, 0.1389, 0.2371, 0.0691, 0.1072, 0.1099, 0.0673, 0.1135]],
    "SAGE":     [[0.9356, 0.9024, 0.9559, 0.7855, 0.8233, 0.8432, 0.8633, 0.9007, 0.8682, 0.8805, 0.9326, 0.8985, 0.8543, 0.7829, 0.8904],
                 [0.6322, 0.6548, 0.4611, 0.2977, 0.4124, 0.4530, 0.3408, 0.2277, 0.4290, 0.6223, 0.2074, 0.3922, 0.5108, 0.2239, 0.4013]],
    "ChebNet":  [[0.9463, 0.9097, 0.9563, 0.8201, 0.8858, 0.8212, 0.8763, 0.8792, 0.8918, 0.9207, 0.8301, 0.9064, 0.8795, 0.7920, 0.8454],
                 [0.6492, 0.6529, 0.4391, 0.3673, 0.4355, 0.4187, 0.3066, 0.3178, 0.4010, 0.6577, 0.3011, 0.5828, 0.5419, 0.2076, 0.3815]],
    "MTGCN":    [[0.9434, 0.9073, 0.9638, 0.8197, 0.8951, 0.8339, 0.8822, 0.9607, 0.8925, 0.8923, 0.8803, 0.9153, 0.9125, 0.8142, 0.8881],
                 [0.6265, 0.6586, 0.5429, 0.3842, 0.4818, 0.4308, 0.3690, 0.3776, 0.4653, 0.5949, 0.2793, 0.5670, 0.5423, 0.2823, 0.4576]],
    "MNGCL":    [[0.9481, 0.9156, 0.8909, 0.8344, 0.8745, 0.8261, 0.8349, 0.8748, 0.9035, 0.9064, 0.8599, 0.9653, 0.8821, 0.8437, 0.9029],
                 [0.6132, 0.7115, 0.4433, 0.4068, 0.3722, 0.4413, 0.3069, 0.2111, 0.4727, 0.5751, 0.2146, 0.6488, 0.4995, 0.2993, 0.4151]],
    "Emogi":    [[0.9063, 0.8951, 0.9596, 0.7821, 0.8345, 0.7762, 0.8727, 0.8848, 0.8308, 0.8228, 0.7691, 0.9044, 0.8259, 0.7448, 0.8545],
                 [0.5375, 0.6329, 0.4287, 0.3437, 0.3610, 0.3576, 0.2834, 0.1530, 0.3565, 0.4798, 0.1640, 0.5392, 0.3782, 0.2206, 0.3444]],
    "ECD-CDGI": [[0.9354, 0.8973, 0.9712, 0.7964, 0.8836, 0.8047, 0.8738, 0.9366, 0.8719, 0.8779, 0.8635, 0.8995, 0.9003, 0.8043, 0.8826],
                 [0.5810, 0.6451, 0.4297, 0.3521, 0.4296, 0.3896, 0.3075, 0.3073, 0.4398, 0.5588, 0.2303, 0.5397, 0.4928, 0.2641, 0.3952]],
    "DISHyper": [[0.8977, 0.1141, 0.9409, 0.8012, 0.8870, 0.8223, 0.8431, 0.8700, 0.8406, 0.8315, 0.8683, 0.8917, 0.8782, 0.8524, 0.8493],
                 [0.4293, 0.0468, 0.3673, 0.3374, 0.3645, 0.4254, 0.2831, 0.1433, 0.3497, 0.4786, 0.1828, 0.4539, 0.4372, 0.3110, 0.3003]],
    "DISFusion":[[0.9366, 0.9132, 0.9429, 0.8073, 0.8587, 0.8262, 0.8941, 0.9183, 0.8851, 0.8653, 0.9238, 0.9023, 0.8840, 0.8221, 0.8812],
                 [0.5857, 0.6928, 0.5012, 0.3610, 0.4036, 0.4525, 0.3129, 0.1828, 0.4813, 0.5585, 0.2919, 0.5075, 0.4625, 0.2881, 0.4093]],
    "BIMODriver":[[0.9518, 0.9257, 0.9899, 0.8304, 0.8972, 0.8452, 0.9099, 0.9648, 0.9040, 0.8852, 0.9054, 0.9363, 0.9193, 0.8347, 0.9067],
                 [0.6629, 0.7123, 0.6438, 0.3910, 0.4826, 0.4672, 0.3532, 0.3701, 0.4575, 0.6234, 0.3298, 0.6226, 0.5934, 0.3010, 0.4618]]
}

if __name__ == "__main__":
    comparator = MethodComparator()
    for name, metrics in data.items():
        comparator.add_method(name, metrics[0], metrics[1])
    
    # 执行比较
    comparator.compare_against(target_method='BIMODriver')

In [ ]:
# ===== Benjamini-Hochberg FDR 校正（并列 p 取平均秩 → 单调化 → 截断 1） =====
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

def bh_fdr(pvals):
    p = np.asarray(list(pvals), dtype=float)
    out = np.full(p.shape, np.nan)
    finite = np.isfinite(p)
    pv = p[finite]
    n = pv.size
    if n == 0:
        return out
    order = np.argsort(pv, kind="mergesort")
    ranked = pv[order]
    ranks = np.arange(1, n + 1, dtype=float)
    i = 0
    while i < n:
        j = i
        while j + 1 < n and ranked[j + 1] == ranked[i]:
            j += 1
        ranks[i:j + 1] = (i + 1 + j + 1) / 2.0
        i = j + 1
    q = ranked * n / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.minimum(q, 1.0)
    out_sorted = np.empty(n)
    out_sorted[order] = q
    out[finite] = out_sorted
    return out

def wilcoxon_one_sided(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    try:
        return float(wilcoxon(x, y, alternative="greater").pvalue)
    except ValueError:
        return float("nan")

def build_bh_table(records, group_col="metric"):
    df = pd.DataFrame(records)
    df["q_BH_merged"] = bh_fdr(df["p"].values)
    df["q_BH_by_metric"] = np.nan
    for metric, idx in df.groupby(group_col, sort=False).groups.items():
        df.loc[idx, "q_BH_by_metric"] = bh_fdr(df.loc[idx, "p"].values)
    df["significant_05"] = df["q_BH_merged"] < 0.05
    return df


# ---------- CPDB / STRING：直接使用 cells 1/2 末尾保存的 results_cpdb / results_string ----------
for label, res in [("CPDB", globals().get("results_cpdb")),
                   ("STRING", globals().get("results_string"))]:
    if res is None:
        print(f"[跳过] {label}: 未找到 results_{label.lower()}（请确认 cells 1/2 已运行并保存）")
        continue
    rec = []
    for metric, m in [("AUC", "auc"), ("AUPRC", "auprc")]:
        for baseline, info in res[m].items():
            rec.append({"metric": metric, "baseline": baseline, "p": info["p_value"]})
    tab = build_bh_table(rec, group_col="metric")
    print(f"\n=== {label}：{len(tab)} 个检验（9 基线 × 2 指标）===")
    print(tab.to_string(index=False, float_format=lambda x: f"{x:.4e}"))
    print(f"合并校正 q<0.05：{tab['significant_05'].sum()} / {len(tab)}")


# ---------- 单癌症 15-均值配对（论文 appendix A9 口径） ----------
PAPER_A9 = {
    ("GAT","AUC"):3.05e-5, ("GAT","AUPRC"):3.05e-5,
    ("SAGE","AUC"):4.27e-4, ("SAGE","AUPRC"):3.05e-5,
    ("ChebNet","AUC"):1.31e-3, ("ChebNet","AUPRC"):2.14e-4,
    ("MTGCN","AUC"):2.14e-4, ("MTGCN","AUPRC"):4.18e-3,
    ("MNGCL","AUC"):3.19e-2, ("MNGCL","AUPRC"):2.69e-3,
    ("Emogi","AUC"):3.05e-5, ("Emogi","AUPRC"):3.05e-5,
    ("ECD-CDGI","AUC"):3.05e-5, ("ECD-CDGI","AUPRC"):3.05e-5,
    ("DISHyper","AUC"):9.16e-5, ("DISHyper","AUPRC"):6.10e-5,
    ("DISFusion","AUC"):3.05e-4, ("DISFusion","AUPRC"):2.14e-4,
}
baselines = [k for k in data.keys() if k != "BIMODriver"]
my_auc = np.array(data["BIMODriver"][0]); my_auprc = np.array(data["BIMODriver"][1])
rec = []
for b in baselines:
    for metric, my, other_idx in [("AUC", my_auc, 0), ("AUPRC", my_auprc, 1)]:
        other = np.array(data[b][other_idx])
        rec.append({"metric": metric, "baseline": b,
                    "p": wilcoxon_one_sided(my, other),
                    "paper_A9": PAPER_A9[(b, metric)]})
df15 = pd.DataFrame(rec)
df15["q_BH_merged"] = bh_fdr(df15["p"].values)
df15["q_BH_by_metric"] = np.nan
for metric, idx in df15.groupby("metric", sort=False).groups.items():
    df15.loc[idx, "q_BH_by_metric"] = bh_fdr(df15.loc[idx, "p"].values)
df15["与论文A9一致(rtol=0.05)"] = df15.apply(
    lambda r: bool(np.isclose(r["p"], r["paper_A9"], rtol=0.05, atol=1e-7)) if not np.isnan(r["p"]) else False,
    axis=1)
df15["BH_显著(q<0.05)"] = df15["q_BH_merged"] < 0.05

print("\n=== 单癌症 15-均值配对（与论文 appendix A9 对照）===")
print(df15.to_string(index=False, float_format=lambda x: f"{x:.4e}"))
print(f"\n与论文 A9 一致：{df15['与论文A9一致(rtol=0.05)'].sum()} / {len(df15)}")
print(f"BH 合并校正 q<0.05：{df15['BH_显著(q<0.05)'].sum()} / {len(df15)}")
print(f"BH 分指标校正 q<0.05：{(df15['q_BH_by_metric']<0.05).sum()} / {len(df15)}")


: 